In [1]:

!pip install requests beautifulsoup4 pandas matplotlib


In [2]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
from urllib.parse import urljoin

print("All libraries imported successfully!")


All libraries imported successfully!


In [3]:

url = "https://books.toscrape.com/"

response = requests.get(url, timeout=20)

print("Status code:", response.status_code)
print("Page downloaded:", len(response.text), "characters")


Status code: 200
Page downloaded: 51294 characters


In [4]:
response = requests.get(url, timeout=20)

In [5]:
print(response.text)


<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" />
        <meta name="created" content="24th Jun 2016 09:29" />
        <meta name="description" content="" />
        <meta name="viewport" content="width=device-width" />
        <meta name="robots" content="NOARCHIVE,NOCACHE" />

        <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
        <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->

        
            <link rel="shortcut icon" href="static/oscar/favicon.

In [6]:

first_page_url = urljoin(
    url,
    "catalogue/page-1.html"
)

response = requests.get(first_page_url, timeout=20)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

book_cards = soup.select("article.product_pod")

print("Number of books found:", len(book_cards))
print(book_cards[0].prettify()[:1500])


Number of books found: 20
<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [7]:

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

print(rating_map)


{'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}


In [8]:

first_book = book_cards[0]

title_element = first_book.select_one("h3 a")

title = title_element["title"]

print("Book title:", title)


Book title: A Light in the Attic


In [9]:

price_element = first_book.select_one("p.price_color")

price_text = price_element.get_text(strip=True)

price = float(
    price_text.replace("£", "").replace("Â", "")
)

print("Original price:", price_text)
print("Numeric price:", price)


Original price: Â£51.77
Numeric price: 51.77


In [10]:

rating_element = first_book.select_one("p.star-rating")

rating_classes = rating_element["class"]

print("Rating HTML classes:", rating_classes)


Rating HTML classes: ['star-rating', 'Three']


In [11]:

rating_word = next(
    word for word in rating_classes
    if word in rating_map
)

rating = rating_map[rating_word]

print("Rating word:", rating_word)
print("Numeric rating:", rating)


Rating word: Three
Numeric rating: 3


In [12]:

availability_element = first_book.select_one(
    "p.instock.availability"
)

availability_text = availability_element.get_text(
    " ",
    strip=True
)

availability = (
    "In stock"
    if "In stock" in availability_text
    else "Out of stock"
)

print("Original availability:", availability_text)
print("Clean availability:", availability)


Original availability: In stock
Clean availability: In stock


In [13]:

one_book = {
    "Title": title,
    "Price": price,
    "Rating": rating,
    "Availability": availability
}

print(one_book)


{'Title': 'A Light in the Attic', 'Price': 51.77, 'Rating': 3, 'Availability': 'In stock'}


In [14]:

records = []

for book in book_cards:

    # Extract title
    title = book.select_one("h3 a")["title"]

    # Extract price
    price_text = book.select_one(
        "p.price_color"
    ).get_text(strip=True)

    price = float(
        price_text.replace("£", "").replace("Â", "")
    )

    # Extract rating
    rating_classes = book.select_one(
        "p.star-rating"
    )["class"]

    rating_word = next(
        word for word in rating_classes
        if word in rating_map
    )

    rating = rating_map[rating_word]

    # Extract availability
    availability_text = book.select_one(
        "p.instock.availability"
    ).get_text(" ", strip=True)

    availability = (
        "In stock"
        if "In stock" in availability_text
        else "Out of stock"
    )

    # Store the record
    records.append({
        "Title": title,
        "Price": price,
        "Rating": rating,
        "Availability": availability
    })

print("Total records extracted:", len(records))
print("\nFirst record:")
print(records[0])


Total records extracted: 20

First record:
{'Title': 'A Light in the Attic', 'Price': 51.77, 'Rating': 3, 'Availability': 'In stock'}


In [15]:

df = pd.DataFrame(records)

display(df.head())


,Title,Price,Rating,Availability
0,A Light in the Attic,51.77,3,In stock
1,Tipping the Velvet,53.74,1,In stock
2,Soumission,50.10,1,In stock
3,Sharp Objects,47.82,4,In stock
4,Sapiens: A Brief History of Humankind,54.23,5,In stock


In [16]:

print("Rows and columns:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)


Rows and columns: (20, 4)

Column names: ['Title', 'Price', 'Rating', 'Availability']

Data types:
Title            object
Price           float64
Rating            int64
Availability     object
dtype: object


In [17]:

df.to_csv("competitor_books.csv", index=False)

print("First-page dataset saved successfully!")


First-page dataset saved successfully!


In [18]:

from google.colab import files

files.download("competitor_books.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/"

RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

print("Scraper settings are ready!")


Scraper settings are ready!


In [20]:

def scrape_page(page_number):
    """Extract book information from one catalogue page."""

    page_url = urljoin(
        BASE_URL,
        f"catalogue/page-{page_number}.html"
    )

    response = requests.get(
        page_url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    book_cards = soup.select("article.product_pod")

    page_records = []

    for book in book_cards:

        # Extract title
        title = book.select_one("h3 a")["title"]

        # Extract and clean price
        price_text = book.select_one(
            "p.price_color"
        ).get_text(strip=True)

        price = float(
            price_text.replace("£", "").replace("Â", "")
        )

        # Convert written rating into a number
        rating_classes = book.select_one(
            "p.star-rating"
        )["class"]

        rating_word = next(
            word for word in rating_classes
            if word in RATING_MAP
        )

        rating = RATING_MAP[rating_word]

        # Extract availability
        availability_text = book.select_one(
            "p.instock.availability"
        ).get_text(" ", strip=True)

        availability = (
            "In stock"
            if "In stock" in availability_text
            else "Out of stock"
        )

        # Extract the individual book's webpage
        relative_url = book.select_one("h3 a")["href"]
        product_url = urljoin(page_url, relative_url)

        # Store this book as a dictionary
        page_records.append({
            "Title": title,
            "Price": price,
            "Rating": rating,
            "Availability": availability,
            "Product_URL": product_url
        })

    return page_records

print("Reusable scraping function defined successfully!")


Reusable scraping function defined successfully!


In [21]:

test_records = scrape_page(1)

print("Books extracted:", len(test_records))
print("\nFirst book:")
print(test_records[0])


Books extracted: 20

First book:
{'Title': 'A Light in the Attic', 'Price': 51.77, 'Rating': 3, 'Availability': 'In stock', 'Product_URL': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


In [22]:

all_records = []

for page_number in range(1, 4):

    page_records = scrape_page(page_number)

    all_records.extend(page_records)

    print(
        f"Page {page_number}: "
        f"{len(page_records)} books scraped"
    )

print("\nTotal books collected:", len(all_records))


Page 1: 20 books scraped
Page 2: 20 books scraped
Page 3: 20 books scraped

Total books collected: 60


In [23]:

df = pd.DataFrame(all_records)

# Clean and validate the data
df["Title"] = df["Title"].str.strip()
df["Price"] = pd.to_numeric(df["Price"], errors="raise")
df["Rating"] = pd.to_numeric(df["Rating"], errors="raise")

print("Dataset preview:")
display(df.head())

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate titles:", df["Title"].duplicated().sum())


Dataset preview:


,Title,Price,Rating,Availability,Product_URL
0,A Light in the Attic,51.77,3,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,1,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,1,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,4,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,https://books.toscrape.com/catalogue/sapiens-a...


Dataset shape: (60, 5)

Column names:
['Title', 'Price', 'Rating', 'Availability', 'Product_URL']

Missing values:
Title           0
Price           0
Rating          0
Availability    0
Product_URL     0
dtype: int64

Duplicate titles: 0


In [24]:

df.to_csv("competitor_books.csv", index=False)

print("Three-page dataset saved successfully!")
print("Total rows saved:", len(df))


Three-page dataset saved successfully!
Total rows saved: 60


In [25]:

from google.colab import files
files.download("competitor_books.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>